In [5]:
import os

from nwtrace import *
import pandas as pd
import geopandas as gpd
import numpy as np

from pathlib import Path

In [6]:
york_sewers = Path('data/more/york_region_storm.gpkg')

segments = gpd.read_file(york_sewers, layer="segments")
nodes = gpd.read_file(york_sewers, layer="nodes")

nodes = nodes[["FACILITYID", "AVG_ELEV", "geometry"]]
segments = segments[["FACILITYID", "geometry"]]

In [3]:
network = utils.network_from_geometry(
    segments=segments,
    nodes=nodes,
    segment_id_field="FACILITYID",
    node_id_field="FACILITYID",
    distance_threshold=0.6
)

In [4]:
repaired_network = utils.verify_flow_directionality(
    segments=network,
    nodes=nodes,
    segment_id_field="FACILITYID",
    elevation_field="AVG_ELEV",
    upstream_field="from",
    downstream_field="to",
    repair_errors=True
)

In [5]:
# real_network = network.dropna(subset=['to']).reset_index()

# real_network.to_csv("data/more/raw_vaughan_network.csv")

network_gdf = network.merge(segments, on="FACILITYID")
network_gdf = gpd.GeoDataFrame(
    network_gdf,
    crs=segments.crs
)

In [6]:
network.set_index("FACILITYID").loc["STMSS88491"]

role
from           STMMH864
to                  NaN
from_dist           0.0
to_dist             NaN
from_height       204.0
to_height           NaN
Name: STMSS88491, dtype: object

In [7]:
repaired_network.set_index("FACILITYID").loc["STMSL10057"]

role
from           STMMH5443
to             STMMH5444
from_dist            0.0
to_dist              0.0
from_height     177.5815
to_height        176.066
Name: STMSL10057, dtype: object

In [8]:
def fix_missing_nodes(
    segment_network,
    segment_id_field,
    downstream_field,
    distance_threshold = 0.1
):
    
    # get just dead end segments from the network (no downstream node)
    dead_ends = segment_network[pd.isna(segment_network[downstream_field])].reset_index()

    # get nearby segments
    nearby_segs = utils.find_nearby_nodes(
        dead_ends, 
        segment_network, 
        segment_id_field, 
        distance_threshold=distance_threshold
    )

    nearby_segs_filtered = nearby_segs[nearby_segs["role"] == downstream_field]
    # remove segments where the closest segment is itself
    nearby_segs_filtered = nearby_segs_filtered[nearby_segs_filtered[segment_id_field] != nearby_segs_filtered["segment_id"]]
    # select the first of the closest potential connection
    best_candidates = (
            nearby_segs_filtered
            .sort_values("dist")
            .groupby(["segment_id"], as_index=False)
            .head(1)
        )
    
    # refactor candidates to represent the new segment-node connection
    repair_gdf = best_candidates[["segment_id", downstream_field]].rename(columns={"segment_id": segment_id_field})
    repair_gdf = repair_gdf.set_index(segment_id_field)

    network_indexed = segment_network.set_index(segment_id_field)

    network_indexed.update(repair_gdf)
    repaired_network = network_indexed.reset_index()

    return repaired_network

In [14]:
def fix_all_missing_nodes(
    segment_network,
    segment_id_field="FACILITYID",
    downstream_field="to",
    distance_threshold=0.1,
    max_iterations=10
):
    current = segment_network.copy()

    for i in range(max_iterations):
        # run one pass of your existing function
        repaired = fix_missing_nodes(
            current,
            segment_id_field=segment_id_field,
            downstream_field=downstream_field,
            distance_threshold=distance_threshold
        )

        # check if anything changed
        missing_before = current[downstream_field].isna().sum()
        missing_after = repaired[downstream_field].isna().sum()

        # update the working network
        current = repaired

        # stop if no missing nodes remain
        if missing_after == 0:
            break

        # stop if the iteration made no progress
        if missing_after == missing_before:
            break

    return i, current

In [24]:

iterations, func_repaired_network = fix_all_missing_nodes(
    network_gdf,
    segment_id_field="FACILITYID",
    downstream_field="to",
    distance_threshold=0.1,
    max_iterations=10
)

In [25]:
iterations

3

In [35]:
real_network = func_repaired_network.dropna(subset="to")

real_network.to_csv("data/more/repaired_vaughan_network.csv")